# 🎙️ Chatterbox TTS — Colab GPU Server

This notebook runs Chatterbox Multilingual V3 on a free Colab T4 GPU
and exposes it over ngrok so your local VoiceLink backend can call it remotely.

## Quick Start
1. **Runtime → Change runtime type → T4 GPU**
2. **Run all cells** (Runtime → Run all)
3. Copy the **Public URL** printed at the bottom (e.g. `https://xxxx.ngrok-free.app`)
4. Paste it into your local `.env` as `COLAB_TTS_URL=https://xxxx.ngrok-free.app`

⚠️ **The URL changes every time you restart the runtime.** Re-run all cells and update `.env`.

## Cell 1 — Install dependencies

In [ ]:
# Install chatterbox-tts, FastAPI, uvicorn, pyngrok, and audio libs.
# This takes ~2-3 min on first run.
# Install chatterbox-tts with --no-deps first (to avoid torch downgrade),
# then install remaining deps separately.
!pip install -q chatterbox-tts --no-deps
!pip install -q fastapi uvicorn pyngrok python-multipart soundfile numpy huggingface_hub transformers

# Verify torch/torchvision are compatible with the Colab CUDA driver
import torch
print(f"torch {torch.__version__}, CUDA {torch.version.cuda}, available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Cell 2 — Load Chatterbox model on GPU

## Cell 3 — FastAPI server + ngrok tunnel

In [ ]:
import base64
import io
import os
import subprocess
import threading
import time

import numpy as np
import soundfile as sf
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

# Ensure model/device from Cell 2 are available in this cell's thread
import torch
from chatterbox.mtl_tts import ChatterboxMultilingualTTS as _CMT
if 'model' not in dir() or model is None:
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    model = _CMT.from_pretrained(device=_device)
else:
    _device = "cuda" if torch.cuda.is_available() else "cpu"
device = _device
print(f"✅ Model ready on {device}")

# Supported languages (Chatterbox Multilingual V3)
SUPPORTED_LANGUAGES = {
    "ar", "da", "de", "el", "en", "es", "fi", "fr", "he", "hi",
    "it", "ja", "ko", "ms", "nl", "no", "pl", "pt", "ru", "sv",
    "sw", "tr", "zh",
}

app = FastAPI(title="Chatterbox TTS Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


class SynthesizeRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=5000)
    reference_audio_b64: str = Field(..., description="Base64-encoded WAV reference audio")
    language: str = Field("en", description="ISO language code")
    speed: float = Field(1.0, ge=0.5, le=2.0)


@app.get("/health")
def health():
    return {
        "status": "ok",
        "device": device,
        "model_loaded": model is not None,
        "cuda_available": torch.cuda.is_available(),
    }


@app.post("/synthesize")
def synthesize(req: SynthesizeRequest):
    """Synthesize speech with a cloned voice. Same signature as local chatterbox_engine."""
    if req.language not in SUPPORTED_LANGUAGES:
        raise HTTPException(
            status_code=400,
            detail=f"Unsupported language '{req.language}'. Supported: {', '.join(sorted(SUPPORTED_LANGUAGES))}",
        )

    # Decode reference audio
    try:
        ref_bytes = base64.b64decode(req.reference_audio_b64)
        ref_audio, ref_sr = sf.read(io.BytesIO(ref_bytes))
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"Invalid reference audio: {exc}")

    # Save temp WAV for Chatterbox (it wants a file path)
    tmp_path = "/tmp/ref_tmp.wav"
    sf.write(tmp_path, ref_audio, ref_sr, format="WAV")

    t0 = time.time()
    try:
        wav = model.generate(
            text=req.text,
            language_id=req.language,
            audio_prompt_path=tmp_path,
        )
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Model inference failed: {exc}")
    elapsed = time.time() - t0

    # Convert tensor → WAV bytes
    if isinstance(wav, torch.Tensor):
        audio_data = wav.cpu().numpy()
    else:
        audio_data = np.asarray(wav, dtype=np.float32)
    if audio_data.ndim > 1:
        audio_data = audio_data.squeeze()
    audio_data = audio_data.astype(np.float32)
    max_val = np.abs(audio_data).max()
    if max_val > 1.0:
        audio_data = audio_data / max_val

    sr = getattr(model, "sr", 24000)
    buf = io.BytesIO()
    sf.write(buf, audio_data, samplerate=sr, format="WAV")
    audio_b64 = base64.b64encode(buf.getvalue()).decode()

    print(f"[SYNTH] text={req.text[:60]!r} lang={req.language} {elapsed:.2f}s", flush=True)
    return {
        "audio_base64": audio_b64,
        "mime_type": "audio/wav",
        "duration_s": len(audio_data) / sr,
        "inference_time_s": round(elapsed, 3),
    }


# ── Start uvicorn in a background thread ──────────────────────────────
def _run_server():
    uvicorn.run(app, host="127.0.0.1", port=8899, log_level="info")

# Kill any existing process on port 8899
import socket
_s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
try:
    _s.connect(('127.0.0.1', 8899))
    _s.close()
    print('⚠️  Port 8899 in use — killing old process...')
    subprocess.run(['fuser', '-k', '8899/tcp'], capture_output=True)
    time.sleep(1)
except ConnectionRefusedError:
    pass  # port is free

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(2)  # let the server bind
print("✅ FastAPI server running on http://127.0.0.1:8899")

# ── Expose via ngrok ──────────────────────────────────────────────────
from pyngrok import ngrok

# ── IMPORTANT: Get your free ngrok auth token ─────────────────────────
# 1. Sign up: https://dashboard.ngrok.com/signup
# 2. Copy token: https://dashboard.ngrok.com/get-started/your-authtoken
# 3. Paste it below (between the quotes):
NGROK_AUTH_TOKEN = "2UvjyzuFvXKDBcyhyMngm8fIAox_4jyHnZnRekwUTU3VcPmUB"
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print("⚠️  No ngrok token set — get one free at https://dashboard.ngrok.com/get-started/your-authtoken")
    print("   Then paste it into NGROK_AUTH_TOKEN above and re-run this cell.")

public_url = ngrok.connect(8899)
url_str = public_url.public_url
print(f"\n{'='*60}")
print(f"🌐 PUBLIC URL: {url_str}")
print(f"{'='*60}")
print(f"\n👉 Paste this into your local .env:")
print(f"   COLAB_TTS_URL={url_str}")
print(f"\n⚠️  This URL changes when you restart the runtime.")

## Cell 4 — Test the endpoint (optional)

In [ ]:
# Quick smoke test: synthesize "Hello world" with a synthetic reference.
import requests
import numpy as np
import soundfile as sf
import io

# Generate a 1-second silent WAV as a dummy reference
dummy = np.zeros(24000, dtype=np.float32)
buf = io.BytesIO()
sf.write(buf, dummy, 24000, format="WAV")
dummy_b64 = base64.b64encode(buf.getvalue()).decode()

resp = requests.post(
    f"http://127.0.0.1:8899/synthesize",
    json={
        "text": "Hello world, this is a test.",
        "reference_audio_b64": dummy_b64,
        "language": "en",
    },
    timeout=60,
)
print(f"Status: {resp.status_code}")
print(f"Response: {resp.json()}")